In [1]:

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
ss = hf.settings_dict()

In [2]:
tmin = 0.7
tmax = 3.7

In [3]:
for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-ret-ref-vol.stc"
        stc = mne.read_source_estimate(hilbert_stc_file)
        stc.crop(0.7, 3.7)
        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        z_stats = np.zeros(n_voxels)
        p_vals = np.zeros(n_voxels)

        # Rayleigh test to get z-scores

        for voxel_idx in range(n_voxels):
            result = rayleigh_test(stc.data[voxel_idx])    # angles in radians
            z_stats[voxel_idx] = result.z       # Z = n * r^2
            p_vals[voxel_idx] = result.pval

        z_stc       = stc.copy().crop(0.7, 0.7 + 1/ss['fs'])
        z_stc.data  = z_stats[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)

        print(f"z min  : {z_stats.min():.4f}")
        print(f"z max  : {z_stats.max():.4f}")
        print(f"z mean : {z_stats.mean():.4f}")
        print(f"Voxels > 0   : {(z_stats > 0).sum()} / {n_voxels}")



loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : 0.1631
z max  : 6001.0000
z mean : 1197.5746
Voxels > 0   : 1440 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : 1.1164
z max  : 6001.0000
z mean : 1278.5764
Voxels > 0   : 1440 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : 0.7034
z max  : 6001.0000
z mean : 1076.5808
Voxels > 0   : 1440 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : 2.3127
z max  : 6001.0000
z mean : 1742.1006
Voxels > 0   : 1440 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : 2.8933
z max  : 6001.0000
z mean : 1462.4169
Voxels 